In [1]:
import glob
import os

import numpy as np
import pandas as pd

os.makedirs("../submission", exist_ok=True)

files = sorted(glob.glob("../data/*.csv.zip"))
symbols = [os.path.basename(f).replace(".csv.zip", "") for f in files]

quotes = [pd.read_csv(f, compression="zip") for f in files]
open_prices = np.vstack([q["open"] for q in quotes])
close_prices = np.vstack([q["close"] for q in quotes])
len(symbols), open_prices.shape

(56, (56, 1258))

In [2]:
# Daily variations carry most of the information about co-movement
variation = close_prices - open_prices

X = variation.copy().T
X /= X.std(axis=0)

In [3]:
# Sparse inverse covariance: structure of conditional dependencies
from sklearn.covariance import GraphicalLassoCV

edge_model = GraphicalLassoCV()
edge_model.fit(X)

/Users/druiz35/Documents/uni_stuff/predictiva/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


GraphicalLassoCV()

In [4]:
# Clustering with affinity propagation over the estimated covariance
from sklearn.cluster import affinity_propagation

_, labels = affinity_propagation(edge_model.covariance_, random_state=0)
n_labels = labels.max()

names = np.array(symbols)
for i in range(n_labels + 1):
    print(f"Cluster {i + 1}: {', '.join(names[labels == i])}")

Cluster 1: AAPL, AMZN, YHOO
Cluster 2: CMCSA, CVC, TWX
Cluster 3: COP, CVX, TOT, VLO, XOM
Cluster 4: CSCO, DELL, HPQ, IBM, MSFT, SAP, TXN
Cluster 5: BA, GD, NOC, RTN
Cluster 6: AIG, AXP, BAC, CAT, CVS, DD, F, GE, GS, HD, JPM, MAR, MMM, R, WFC, WMT
Cluster 7: MCD
Cluster 8: GSK, NVS, PFE, SNY, UN
Cluster 9: K, KO, PEP
Cluster 10: CL, KMB, PG
Cluster 11: CAJ, HMC, NAV, SNE, TM, XRX


In [5]:
# 2D embedding for visualization
from sklearn.manifold import LocallyLinearEmbedding

node_position_model = LocallyLinearEmbedding(
    n_components=2, eigen_solver="dense", n_neighbors=6
)
embedding = node_position_model.fit_transform(X.T).T

In [6]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

plt.figure(1, facecolor="w", figsize=(12, 9))
plt.clf()
ax = plt.axes([0.0, 0.0, 1.0, 1.0])
plt.axis("off")

# Partial correlations
partial_correlations = edge_model.precision_.copy()
d = 1 / np.sqrt(np.diag(partial_correlations))
partial_correlations *= d
partial_correlations *= d[:, np.newaxis]
non_zero = np.abs(np.triu(partial_correlations, k=1)) > 0.02

plt.scatter(
    embedding[0], embedding[1], s=100 * d**2, c=labels, cmap=plt.cm.nipy_spectral
)

start_idx, end_idx = np.where(non_zero)
segments = [
    [embedding[:, start], embedding[:, stop]] for start, stop in zip(start_idx, end_idx)
]
values = np.abs(partial_correlations[non_zero])
lc = LineCollection(
    segments, zorder=0, cmap=plt.cm.hot_r, norm=plt.Normalize(0, 0.7 * values.max())
)
lc.set_array(values)
lc.set_linewidths(15 * values)
ax.add_collection(lc)

for index, (name, label, (x, y)) in enumerate(zip(names, labels, embedding.T)):
    dx = x - embedding[0]
    dx[index] = 1
    dy = y - embedding[1]
    dy[index] = 1
    this_dx = dx[np.argmin(np.abs(dy))]
    this_dy = dy[np.argmin(np.abs(dx))]
    if this_dx > 0:
        horizontalalignment = "left"
        x = x + 0.002
    else:
        horizontalalignment = "right"
        x = x - 0.002
    if this_dy > 0:
        verticalalignment = "bottom"
        y = y + 0.002
    else:
        verticalalignment = "top"
        y = y - 0.002
    plt.text(
        x,
        y,
        name,
        size=10,
        horizontalalignment=horizontalalignment,
        verticalalignment=verticalalignment,
        bbox=dict(
            facecolor="w",
            edgecolor=plt.cm.nipy_spectral(label / float(n_labels)),
            alpha=0.6,
        ),
    )

plt.xlim(
    embedding[0].min() - 0.15 * np.ptp(embedding[0]),
    embedding[0].max() + 0.10 * np.ptp(embedding[0]),
)
plt.ylim(
    embedding[1].min() - 0.03 * np.ptp(embedding[1]),
    embedding[1].max() + 0.03 * np.ptp(embedding[1]),
)

plt.savefig("../submission/stocks.png", dpi=120, bbox_inches="tight")
plt.show()

<Figure size 1200x900 with 1 Axes>